In [1]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from cmcrameri import cm
import cartopy
import cartopy.crs as ccrs
from cartopy.io.shapereader import Reader
import xesmf as xe # for regridding data
import rioxarray
import gc # garbage collection

In [ ]:
# make nice plots
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['figure.autolayout'] = False
plt.rcParams['font.size'] = 14
##plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300  # Set default DPI to 300 for poster figures

In [ ]:
# coordinates for region of interest
min_lat_plotting = -22.
max_lat_plotting = 12.
min_lon_plotting = 360.-82.
max_lon_plotting = 360.-42.

geog_range_plotting = [min_lon_plotting, max_lon_plotting, min_lat_plotting, max_lat_plotting]
print(geog_range_plotting)

In [ ]:
# shapefile for Amazon ecoregion
Amazon_shapefile='/global/homes/j/jkowalcz/shapefileamazonecoregionwwf/amazon_ecolola.shp'


In [ ]:
## Look at one GeoTIFF file from CTrees to see what coordinate system is

## code by Google Gemini

# 1. Load the file (lazily, so it doesn't eat your RAM)
file_path = "/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/CTrees_global_AGCB/ctrees_global_2000_AGC.tif"
rds = rioxarray.open_rasterio(file_path, chunks=True)

# 2. Print the spatial "ID Card"
print(f"--- Spatial Metadata for {file_path} ---")
print(f"Coordinate Reference System (CRS): {rds.rio.crs}")
print(f"Bounds (minx, miny, maxx, maxy): {rds.rio.bounds()}")
print(f"Width/Height: {rds.rio.width} x {rds.rio.height}")
print(f"Number of Bands: {rds.rio.count}")
print(f"Resolution (pixel size): {rds.rio.resolution()}")

# 3. Quick Visual Look
# We 'coarsen' the data by a factor of 20 just for the plot 
# so it displays instantly.
print("\nGenerating low-res preview...")
rds.sel(band=1).coarsen(x=20, y=20, boundary='trim').mean().plot(cmap="viridis")

plt.title("Low-Res Preview of GeoTIFF")
plt.show()

In [ ]:
## subset files for Amazon and average over 2000-2014
## code by Google Gemini

def average_regional_subset(input_dir, output_file, bounds):
    """
    Masks -9999.0 to NaN, subsets by bounds, and averages across all files.
    """
    tiff_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('_AGC.tif')]
    
    if not tiff_files:
        print("No files found.")
        return

    subset_list = []
    
    for f in tiff_files:
        # 1. Open lazily
        ds = rioxarray.open_rasterio(f, chunks={'x': 1000, 'y': 1000})
        
        # 2. Subset first (saves memory by masking a smaller area)
        subset = ds.rio.clip_box(*bounds)
        
        # 3. Replace -9999.0 with NaN
        # This converts the data to float if it wasn't already
        subset = subset.where(subset != -9999.0)
        subset_list.append(subset)

    # 4. Stack and Mean
    stack = xr.concat(subset_list, dim="stack")
    regional_average = stack.mean(dim="stack", skipna=True)
    
    # 5. Save
    # Note: Added os.path.join for cleaner path handling
    full_output_path = os.path.join(input_dir, output_file)
    regional_average.rio.to_raster(full_output_path)
    
    print(f"Regional average (masked and subsetted) saved to: {full_output_path}")

# Example usage for the Amazon:
# amazon_bounds = (-80, -20, -35, 10) # rough minx, miny, maxx, maxy
# average_regional_subset('/path/to/tiffs/', 'amazon_average_2000_2014.tif', amazon_bounds)

In [3]:
# minx, miny, maxx, maxy
bounds=(-82,-22,-42,12)
path='/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/CTrees_global_AGCB/'
outfile='regional_average.tif'

In [ ]:
### make regionally subset, average file. -- only need to do this once!
#average_regional_subset(path,outfile,bounds)

In [6]:
ctrees = rioxarray.open_rasterio(path+outfile)

In [7]:
# save as a NetCDF file too
ilamb_dir="/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/ILAMB/"
ctrees_lat_lon = ctrees.rename({"x": "lon", "y": "lat"})
ctrees_lat_lon.to_netcdf(ilamb_dir+'CTrees_biomass_Amazon_regional_avg2000-2014.nc')

In [ ]:
## convert MgCO2/ha to MgC/ha
conv=(12./44.)

In [ ]:
ax = plt.subplot(projection=ccrs.PlateCarree())
(conv*ctrees.sel(band=1)).plot(levels=np.linspace(0,300,31),cmap=cm.imola_r,cbar_kwargs=dict(shrink=1,location='right',orientation='vertical',label='AGCB (Mg/Ha)'))
ax.coastlines()
ax.add_geometries(Reader(Amazon_shapefile).geometries(),ccrs.PlateCarree(),edgecolor=(0.35,0.35,0.35,1),facecolor=(0,0,0,0))
plt.title('CTrees AGCB avg 2000-2014')
